# 03 Model B: hERG block

Train the model that answers **condition 3** of the screen: does this drug block the hERG potassium channel? It learns from the TDC `hERG` set, which is only a few hundred compounds.

The target is ROC-AUC >= 0.85. Model B is only a **fallback**: for drugs on the CredibleMeds QT list, the clinical listing is used instead of a prediction.

### Why this notebook looks different from notebook 02

Model A had 12,328 compounds, so one held out split of 2,466 molecules gave a stable number. Here there are 655 compounds. A single 20 percent holdout is just 131 molecules, and the score would swing by several points depending on the random seed alone. So **everything here is cross validated**, and we report the spread across folds, not only the mean.

In [ ]:
import sys
import os
import warnings

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import features
import models
import applicability

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

PROCESSED_DIR = os.path.join("..", "data", "processed")

## Step 1. Load, and look hard at how little data there is

Two problems are visible straight away, and they shape every choice below.

In [ ]:
X = np.load(os.path.join(PROCESSED_DIR, "X_herg.npy"))
y = np.load(os.path.join(PROCESSED_DIR, "y_herg.npy")).astype(int)
herg_df = pd.read_csv(os.path.join(PROCESSED_DIR, "herg_clean.csv"))
smiles = herg_df["Drug"].tolist()

print("features:", X.shape)
print("blockers:", int(y.sum()), "of", len(y), "=", round(float(y.mean()), 3))
print()
print("Problem 1: only", len(y), "molecules. Model A had 12,328.")
print("Problem 2:", round(100 * float(y.mean()), 1),
      "percent are blockers, so always guessing 'blocker' already scores",
      round(100 * float(y.mean()), 1), "percent accuracy.")
print()
print("Problem 2 is why we report ROC-AUC and never plain accuracy.")

## Step 2. Group the molecules by scaffold

Same idea as notebook 02: strip the side chains off each molecule and keep the ring core.
Here we do not use it to make one split, but to define **groups**, so that cross validation
can keep every molecule sharing a core inside the same fold.

In [ ]:
groups = []
for smi in smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        groups.append("")
        continue
    groups.append(MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False))
groups = np.array(groups)

sizes = pd.Series(groups).value_counts()
print("distinct scaffolds:", len(sizes), "across", len(groups), "molecules")
print("appearing only once:", int((sizes == 1).sum()))
print("largest group:", int(sizes.iloc[0]), "molecules")

## Step 3. Two kinds of cross validation

Cross validation splits the data into 5 parts, trains on 4 and tests on the 5th, then
rotates so every molecule is tested exactly once. We run it two ways, and the difference
between them is the point:

- **Stratified CV** (repeated 3 times, so 15 fits) keeps the blocker/non-blocker ratio
  steady in every fold. It is *stable*, which is what we want for choosing between models.
  But it lets analogues of the same scaffold sit on both sides, so it flatters.
- **Scaffold grouped CV** keeps every shared ring core inside one fold. It is *honest*, and
  it is the number we report.

We choose the model on the stable one and report the honest one. Choosing on the number we
report would be tuning against our own test.

In [ ]:
CANDIDATES = {
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=42),
    "GradientBoosting": lambda: HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.08, random_state=42),
    "GradientBoosting (small)": lambda: HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=42),
    "LogisticRegression": lambda: make_pipeline(
        StandardScaler(with_mean=False),
        LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1)),
}


def cross_validate(make_model, splitter, use_groups):
    """Return the ROC-AUC of every fold."""
    aucs = []
    splits = splitter.split(X, y, groups) if use_groups else splitter.split(X, y)
    for train_rows, test_rows in splits:
        if len(np.unique(y[test_rows])) < 2:
            continue  # a fold with only one class has no ROC-AUC
        clf = make_model()
        clf.fit(X[train_rows], y[train_rows])
        aucs.append(roc_auc_score(y[test_rows], clf.predict_proba(X[test_rows])[:, 1]))
    return np.array(aucs)


stratified = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scaffold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
fold_scores = {}
for name, make_model in CANDIDATES.items():
    strat = cross_validate(make_model, stratified, False)
    scaf = cross_validate(make_model, scaffold, True)
    fold_scores[name] = scaf
    rows.append({
        "model": name,
        "stratified_cv": strat.mean(),
        "stratified_sd": strat.std(),
        "scaffold_cv": scaf.mean(),
        "scaffold_sd": scaf.std(),
    })
    print(f"{name:26s} stratified {strat.mean():.3f} +/- {strat.std():.3f}"
          f"   scaffold {scaf.mean():.3f} +/- {scaf.std():.3f}")

results = pd.DataFrame(rows)

In [ ]:
results_sorted = results.sort_values("stratified_cv", ascending=False)
print("ranked by the STABLE score, which is what we select on:")
print(results_sorted.round(3).to_string(index=False))

best_name = results_sorted.iloc[0]["model"]
best_row = results_sorted.iloc[0]
print()
print("chosen:", best_name)
print("reportable (scaffold) ROC-AUC:", round(best_row["scaffold_cv"], 3),
      "+/-", round(best_row["scaffold_sd"], 3))
print("target >= 0.85 ->",
      "PASS" if best_row["scaffold_cv"] >= 0.85 else "BELOW TARGET")

## Step 4. Show the spread, not just the mean

With this little data the mean alone is misleading. The box plot shows how much each
model's score moves from fold to fold. A model whose folds scatter widely has not really
earned the average it reports.

In [ ]:
plt.figure(figsize=(7.5, 4.2))
names = list(fold_scores.keys())
plt.boxplot([fold_scores[n] for n in names], labels=names, showmeans=True)
for i, n in enumerate(names):
    plt.scatter(np.full(len(fold_scores[n]), i + 1), fold_scores[n],
                alpha=0.6, zorder=3, s=25)
plt.axhline(0.85, color="red", linestyle="--", label="target 0.85")
plt.axhline(0.5, color="grey", linestyle=":", label="coin flip")
plt.ylabel("ROC-AUC (scaffold grouped folds)")
plt.title("Model B: fold by fold spread on 655 compounds")
plt.xticks(rotation=15, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## Step 5. Train the final Model B and save it

Once validation is settled we refit the chosen model on **all** 655 compounds. There is no
separate holdout left at this point, and that is deliberate: the cross validated score above
is the estimate of how it performs, and with a set this small we would rather the shipped
model saw every molecule than keep 131 of them idle.

We also fit an applicability domain for Model B. It matters less than Model A's, because
Model B is pointed at prescription drugs, which is the same kind of chemistry it trained on,
but the screen should still be able to say "this drug is unlike anything I know".

In [ ]:
model_b = CANDIDATES[best_name]()
model_b.fit(X, y)

train_bits = features.bits_from_matrix(X)
domain_b = applicability.fit_domain(train_bits)

joblib.dump(model_b, os.path.join(PROCESSED_DIR, "model_b.joblib"))
joblib.dump(domain_b, os.path.join(PROCESSED_DIR, "domain_b.joblib"))

pd.DataFrame([{
    "model": "B_herg",
    "algorithm": best_name,
    "features": str(features.FP_SIZE) + " counts + " + str(features.N_DESCRIPTORS) + " desc",
    "n_molecules": len(y),
    "positive_fraction": round(float(y.mean()), 4),
    "stratified_cv_auc": round(float(best_row["stratified_cv"]), 4),
    "scaffold_cv_auc": round(float(best_row["scaffold_cv"]), 4),
    "scaffold_cv_sd": round(float(best_row["scaffold_sd"]), 4),
    "target": 0.85,
    "domain_floor": round(float(domain_b["floor"]), 4),
}]).to_csv(os.path.join(PROCESSED_DIR, "model_b_metrics.csv"), index=False)

results.to_csv(os.path.join(PROCESSED_DIR, "model_b_selection.csv"), index=False)
print("saved model_b.joblib, domain_b.joblib, model_b_metrics.csv, model_b_selection.csv")